<a href="https://colab.research.google.com/github/general-molecular-simulations/so3lr/blob/main/examples/so3lr_colab_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install SO3LR
#@markdown ## Set the runtime to GPU (Runtime -> Change runtime type -> GPU), then run this cell. Takes ~1-2 min.
#@markdown Colab ships Python 3.12, which SO3LR requires, so no extra Python setup is needed.
import os
import subprocess

_setup_flag = '/content/.SO3LR_SETUP'


def sh(cmd):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True).returncode


if not os.path.exists(_setup_flag):
    # Colab points pip at an internal constraints file via these env vars; uv
    # tries to read it and errors out, so clear them for this session.
    for _var in ('PIP_CONSTRAINT', 'UV_CONSTRAINT'):
        os.environ.pop(_var, None)

    # uv is a fast pip replacement (recommended by SO3LR); fall back to pip.
    use_uv = sh('pip install -q uv==0.11.26') == 0

    def install(spec):
        if use_uv and sh(f'uv pip install --system -q {spec}') == 0:
            return
        assert sh(f'pip install -q {spec}') == 0, f'install failed: {spec}'

    # GPU build of JAX if a CUDA device is visible, otherwise the CPU build.
    has_gpu = subprocess.run('nvidia-smi -L', shell=True,
                             stdout=subprocess.DEVNULL,
                             stderr=subprocess.DEVNULL).returncode == 0
    if not has_gpu:
        print('\nNo GPU detected -- installing the CPU build. This works, but is '
              'much slower.\nRuntime -> Change runtime type -> GPU, then rerun.\n')
    install('"jax[cuda12]"' if has_gpu else 'jax')
    install('ase py3Dmol')

    # SO3LR pulls its git dependencies (the jax-md fork and jax-pme) from pyproject.
    assert sh('git clone -q https://github.com/general-molecular-simulations/so3lr.git '
              '/content/so3lr') == 0, 'clone failed'
    install('/content/so3lr')

    open(_setup_flag, 'w').close()
    print('\nInstalled. If the check below fails, restart the runtime '
          '(Runtime -> Restart session) and run this cell again.')

# Verify the install and fail clearly if a runtime restart is needed.
try:
    import jax
    import so3lr
    print(f'SO3LR ready. JAX backend: {jax.default_backend()}')
except Exception as e:
    raise RuntimeError(
        'SO3LR is not ready. Restart the runtime and re-run this cell.'
    ) from e


## Running simulations

SO3LR ships a small command-line interface; this notebook wraps it in interactive forms.

1. Run the **Install SO3LR** cell above once per session (set the runtime to GPU first).
2. Use **Run an SO3LR simulation** to optimize a geometry (`opt`), run molecular
   dynamics (`nvt`, `npt`, `nve`), or evaluate energies/forces on a dataset (`eval`).
3. Use **Visualize a trajectory** to animate the result.

Change any field and re-run the cell as often as you like — each run replaces its
own output file.

The bundled example `/content/so3lr/examples/data/ala15_folded.xyz` is a folded Ala15
peptide **in vacuum**, which is why `lr_cutoff` defaults to 100 Å: a gas-phase system
should not have its electrostatics and dispersion truncated. For a periodic box, use
12 Å instead. To run your own system, upload an ASE-readable structure file via the
**Files** panel (the folder icon on the left) and set its path as the input.


In [ ]:
#@title Run an SO3LR simulation
#@markdown Pick a simulation type, point to an input structure, set the parameters, and run.

simulation_type = "nvt" #@param ["opt", "nvt", "npt", "nve", "eval"]
input_file = "/content/so3lr/examples/data/ala15_folded.xyz" #@param {type:"string"}
output_file = "" #@param {type:"string"}
model = "so3lr-tauri-m-2606" #@param ["so3lr-tauri-s-2606", "so3lr-tauri-m-2606", "so3lr-tauri-l-2606"]

#@markdown **Common options** — `lr_cutoff`: 100 for gas phase, 12 for a periodic box
lr_cutoff = 100.0 #@param {type:"number"}
total_charge = 0 #@param {type:"integer"}

#@markdown **Molecular dynamics only** (`nvt` / `npt` / `nve`) — temperature in K, timestep in fs
temperature = 300.0 #@param {type:"number"}
timestep_fs = 0.5 #@param {type:"number"}
md_cycles = 50 #@param {type:"integer"}
md_steps = 20 #@param {type:"integer"}
seed = 0 #@param {type:"integer"}
pressure_bar = 1.0 #@param {type:"number"}

#@markdown **Optimization only** (`opt`) — force convergence in eV/Å
force_conv = 0.05 #@param {type:"number"}

#@markdown **Evaluation only** (`eval`) — comma-separated targets, e.g. `forces,dipole_vec`
eval_targets = "forces" #@param {type:"string"}

import os
import shlex
import subprocess

# NPT needs full 3D periodicity; the bundled peptide is in vacuum. Say so plainly
# rather than letting the run fail with a traceback halfway through.
if simulation_type == "npt":
    import ase.io
    input_pbc = ase.io.read(input_file).get_pbc()
    assert input_pbc.all(), (
        f"{input_file} must be periodic in all three axes for NPT; "
        f"received pbc={input_pbc.tolist()}. Use nvt / nve / opt, or "
        "supply a fully periodic structure with a cell.")

if not output_file:
    stem = os.path.splitext(os.path.basename(input_file))[0]
    output_file = f"{stem}_{simulation_type}.xyz"

common = ["--model", model, "--lr-cutoff", str(lr_cutoff),
          "--total-charge", str(total_charge),
          # Without this, re-running the cell aborts because the output exists.
          "--overwrite"]

if simulation_type == "eval":
    cmd = ["so3lr", "eval", "--datafile", input_file, "--save-to", output_file] + common
    if eval_targets.strip():
        cmd += ["--targets", eval_targets.strip()]
else:
    cmd = ["so3lr", simulation_type, "--input", input_file, "--output", output_file] + common
    if simulation_type == "opt":
        cmd += ["--force-conv", str(force_conv)]
    else:  # nvt / npt / nve
        cmd += ["--temperature", str(temperature), "--dt", str(timestep_fs),
                "--md-cycles", str(md_cycles), "--md-steps", str(md_steps),
                "--seed", str(seed)]
        if simulation_type == "npt":
            cmd += ["--pressure", str(pressure_bar)]

print("Running:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n" + "-" * 70)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print("-" * 70)
if code == 0:
    last_output = output_file  # picked up by the visualization cell
    print(f"Done. Output written to: {output_file}")
else:
    raise subprocess.CalledProcessError(code, cmd)


In [ ]:
#@title Visualize a trajectory
#@markdown Leave blank to use the output of the cell above. Multi-frame files are animated.

trajectory_file = "" #@param {type:"string"}

import os
from io import StringIO

import ase.io
import py3Dmol

trajectory_file = trajectory_file.strip() or globals().get("last_output", "")
assert trajectory_file, "Run a simulation first, or type a path above."
assert os.path.exists(trajectory_file), f"File not found: {trajectory_file}"

frames = list(ase.io.read(trajectory_file, index=":"))
print(f"Loaded {len(frames)} frame(s) from {trajectory_file}")

buf = StringIO()
for atoms in frames:
    ase.io.write(buf, atoms, format="xyz")

view = py3Dmol.view(width=800, height=500)
view.addModel(buf.getvalue(), "xyz", {"multimodel": True})
view.setStyle({"sphere": {"radius": 0.4}, "stick": {"radius": 0.2}})
view.setBackgroundColor("white")
view.zoomTo()
if len(frames) > 1:
    view.animate({"loop": "forward", "interval": 100})
view.show()
